# RTM3D/KM3D ResNet-18 — ONNX edge-runtime feasibility

**Revision:** `RTM3D-KM3D-ONNX-EXPORT-2026-10-07-r1`

This exports the same pinned ResNet-18 checkpoint and fixed `1×3×384×1280` input as the Core ML experiment, but as a standard ONNX graph. It packages PyTorch reference outputs for the same 16 evenly spaced Chen validation images and checks ONNX Runtime CPU parity at the unchanged `2e-4` maximum-error limit.

The exported graph contains raw neural-network heads only. It does not train, evaluate AP, include calibration-dependent 3D decoding, or measure iPhone performance. ONNX is an exchange format; an app still needs a runtime. On iPhone, ONNX Runtime can use the Core ML execution provider, while other devices can use their own supported providers.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess, sys
from google.colab import drive, files
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/mobile_adas3d')
PROJECT_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', PROJECT_URL, str(PROJECT_DIR)], check=True)
origin = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
if origin.startswith('git@github.com:'): origin = 'https://github.com/' + origin.split(':', 1)[1]
if origin.rstrip('/').removesuffix('.git').lower() != 'https://github.com/ali-rt/mobile_adas3d':
    raise RuntimeError(f'Unexpected project origin: {origin!r}')
if not (PROJECT_DIR / '.git').exists(): raise RuntimeError(f'{PROJECT_DIR} is not a Git checkout; choose a fresh path.')
branch = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'branch', '--show-current'], text=True).strip()
if branch != 'main': raise RuntimeError(f'Expected main, found {branch!r}; preserve checkout and use a clean main checkout.')
dirty = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'status', '--porcelain', '--untracked-files=no'], text=True).strip()
if dirty: raise RuntimeError('MobileADAS3D has tracked edits; preserve them and use a clean checkout.')
subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
SCRIPT = PROJECT_DIR / 'scripts/export_rtm3d_km3d_res18_onnx.py'
if not SCRIPT.is_file(): raise FileNotFoundError(f'{SCRIPT} missing after updating main; confirm the ONNX export revision is published.')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
OUTPUT_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/rtm3d_km3d_res18_r1')
RTM3D_REPO = Path('/content/RTM3D_KM3D_R1_20261007')
EXPECTED_COMMIT = '888c379e79d8a6d134f06a9b7d669118679e06dc'
RTM3D_URL = 'https://github.com/Banconxuan/RTM3D.git'
CHECKPOINT_ID = '14ww6mxtitO9aDszZN3ai8N7U1doehvi8'
if not (SPLIT_DIR / 'val.txt').is_file(): raise FileNotFoundError(SPLIT_DIR / 'val.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime', 'opencv-python-headless'], check=True)
import torch, onnx, onnxruntime as ort
print('MobileADAS3D main:', subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('PyTorch:', torch.__version__, '| ONNX:', onnx.__version__, '| ONNX Runtime:', ort.__version__)
print('ONNX Runtime providers:', ort.get_available_providers())

## Prepare the exact RTM3D source, checkpoint, and Chen validation data

This preserves existing modified checkouts, selects a clean pinned source revision, and uses the same verified checkpoint, dataset, split, and preprocessing as the Core ML run. No iPhone connection is needed.

In [ ]:
import subprocess
def inspect_repo(path):
    if not path.exists(): return False, 'missing'
    try:
        head = subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
        origin = subprocess.check_output(['git','-C',str(path),'remote','get-url','origin'],text=True,stderr=subprocess.DEVNULL).strip()
        dirty = subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=all'],text=True).strip()
    except subprocess.CalledProcessError: return False, 'not a Git checkout'
    norm = origin.rstrip('/').removesuffix('.git').lower()
    issues = ([f'commit {head}'] if head != EXPECTED_COMMIT else []) + (['unexpected origin'] if norm != RTM3D_URL.removesuffix('.git').lower() else []) + (['modified or untracked files'] if dirty else [])
    return not issues, '; '.join(issues) if issues else 'clean pinned checkout'
ok, why = inspect_repo(RTM3D_REPO)
if RTM3D_REPO.exists() and not ok:
    original = RTM3D_REPO; suffix = 1
    while True:
        candidate = original.with_name(f'{original.name}_fresh{suffix}')
        valid, _ = inspect_repo(candidate)
        if not candidate.exists() or valid: RTM3D_REPO = candidate; break
        suffix += 1
    print(f'Preserving existing RTM3D checkout ({why}); selected {RTM3D_REPO}')
if not RTM3D_REPO.exists():
    subprocess.run(['git','clone',RTM3D_URL,str(RTM3D_REPO)],check=True)
    subprocess.run(['git','-C',str(RTM3D_REPO),'checkout','--detach',EXPECTED_COMMIT],check=True)
ok, why = inspect_repo(RTM3D_REPO)
if not ok: raise RuntimeError(f'RTM3D source verification failed: {why}')
VAL_IDS = [line.strip() for line in (SPLIT_DIR/'val.txt').read_text().splitlines() if line.strip()]
if len(VAL_IDS) != 3769 or len(set(VAL_IDS)) != 3769: raise RuntimeError(f'Expected 3769 unique Chen val IDs, got {len(VAL_IDS)}')
DATA_CANDIDATES = [Path('/content/kitti_m67'),Path('/content/kitti_m66'),Path('/content/kitti_m65'),Path('/content/kitti_m64'),Path('/content/kitti_m62'),Path('/content/kitti_m61'),Path('/content/kitti'),Path('/content/monodetr_kitti_a2'),Path('/content/drive/MyDrive/datasets/kitti')]
DATASET_ROOT = next((root for root in DATA_CANDIDATES if (root/'training/image_2'/f'{VAL_IDS[0]}.png').is_file() and (root/'training/calib'/f'{VAL_IDS[0]}.txt').is_file()),None)
if DATASET_ROOT is None: raise FileNotFoundError('Could not find KITTI val image and calibration. Add your root to DATA_CANDIDATES.')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
CHECKPOINT = OUTPUT_ROOT/'model_res18_1.pth'
if not CHECKPOINT.is_file():
    subprocess.run([sys.executable,'-m','pip','install','-q','gdown'],check=True)
    subprocess.run([sys.executable,'-m','gdown',f'https://drive.google.com/uc?id={CHECKPOINT_ID}','-O',str(CHECKPOINT)],check=True)
if CHECKPOINT.stat().st_size < 1_000_000: raise RuntimeError('Checkpoint file looks incomplete')
print('RTM3D source:', subprocess.check_output(['git','-C',str(RTM3D_REPO),'rev-parse','HEAD'],text=True).strip())
print('Checkpoint:', CHECKPOINT, CHECKPOINT.stat().st_size, 'bytes')
print('Dataset:', DATASET_ROOT, '| validation images:', len(VAL_IDS))
print('Inputs ready. No phone connection needed.')

## Export ONNX and run ONNX Runtime CPU parity on all 16 fixtures

The export and parity command records a report even when the strict parity gate fails, so the bundle remains available for diagnosis. The gate is not relaxed.

In [ ]:
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
EXPORT_DIR = OUTPUT_ROOT / f'onnx_export_r1_{RUN_TAG}'
LOG_DIR = OUTPUT_ROOT / 'colab_logs'; LOG_DIR.mkdir(parents=True,exist_ok=True)
EXPORT_LOG = LOG_DIR / f'onnx_export_r1_{RUN_TAG}.log'
command = [sys.executable,'-u',str(SCRIPT),'--repo',str(RTM3D_REPO),'--checkpoint',str(CHECKPOINT),'--dataset-root',str(DATASET_ROOT),'--split-dir',str(SPLIT_DIR),'--output-dir',str(EXPORT_DIR)]
print('+',' '.join(command),flush=True)
with EXPORT_LOG.open('w') as log:
    process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line); log.flush()
    return_code = process.wait()
if return_code: raise RuntimeError(f'RTM3D ONNX export failed (exit {return_code}); full log: {EXPORT_LOG}')
EXPORT_REPORT = json.loads((EXPORT_DIR/'rtm3d_onnx_export.json').read_text())
CPU_REPORT = json.loads((EXPORT_DIR/'onnxruntime_cpu_parity.json').read_text())
assert EXPORT_REPORT['export_complete'] and EXPORT_REPORT['onnx_checker_passed']
print('ONNX export complete:', EXPORT_REPORT['onnx_sha256'])
print('ONNX Runtime CPU parity passed:', CPU_REPORT['complete'])
print('CPU parity report:', EXPORT_DIR/'onnxruntime_cpu_parity.json')
print('Durable log:', EXPORT_LOG)
BUNDLE = EXPORT_DIR/'rtm3d_km3d_res18_onnx_export.zip'
print('Download this bundle for Mac/provider review:', BUNDLE)
files.download(str(BUNDLE))

## What this establishes—and what it does not

A passing ONNX Runtime CPU parity report establishes that the exported graph reproduces the raw PyTorch heads within tolerance on these 16 images in that runtime. It does not establish iPhone speed, Neural Engine use, accurate 3D decoding, or performance on Android/NVIDIA/Qualcomm. ONNX Runtime uses execution providers; the Core ML provider on iPhone still dispatches supported operations through Apple Core ML. Device-specific speed and provider coverage must be measured on the actual target.

After export, send the ZIP back here. We can audit it on Mac, then decide whether to test ONNX Runtime with the Core ML provider and/or proceed to calibration-based 3D decode validation.